# Smart Fund Advisor — Notebook 3: Federated Learning Simulation (v2)

**Objective:**  
Simulate on-device federated learning using **Flower (`flwr`)** on the 30 % FL customer split.

### Design (v2 — Incremental Waves + FedProx)
| Property | Value |
|----------|-------|
| FL Framework | Flower `flwr` in-process simulator |
| Clients | One per customer in 30% split (~3,750 devices) |
| Records per client | ≤ 4 (most-recent months) |
| Starting weights | Central model (trained on 70% with FocalLoss) |
| Aggregation | FedAvg (weighted by num_examples) + FedProx (adaptive μ) |
| Privacy | Gaussian DP noise on gradients (σ=1.0) |
| Rounds | 15 total (5 waves × 3 rounds) |
| Local epochs | 5 per round |
| Input features | 15 (including new EMI_Income_Ratio, Savings_Rate, Credit_History_Score) |

### Incremental Learning Waves
Users are onboarded in 5 waves (750 per wave). Each wave adds new users to
the pool and runs 3 FL rounds. The global model pseudo-labels new arrivals.

### Privacy Model
Each client:
1. Clips gradients to L2-norm ≤ C (=1.0)
2. Adds Gaussian noise ~ *N*(0, (σ·C)²), σ=1.0

This provides **(ε, δ)-differential privacy** per client — the server never
sees raw data, only noisy gradient updates.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import json

from config import RISK_FEATURES, RANDOM_SEED, FL_ROUNDS, FL_MIN_CLIENTS
from src.utils import set_seed, plot_fl_history, plot_confusion_matrix, print_full_report
from src.central_model import load_central_model, predict

set_seed(RANDOM_SEED)
print('Setup complete ✓')

Setup complete ✓


In [2]:
# ── 1. Load FL customer split produced in Notebook 02 ──
df_fl = pd.read_csv('models/df_fl_split.csv')
print(f'FL split: {df_fl["Customer_ID"].nunique()} customers, {len(df_fl)} rows')
print('Risk label distribution in FL split:')
print(df_fl['risk_label'].value_counts())

FL split: 4375 customers, 4375 rows
Risk label distribution in FL split:
risk_label
Low          1141
High         1086
Medium       1051
Very_High     556
Very_Low      541
Name: count, dtype: int64


In [3]:
# ── 2. Quick client-size audit ──
# Each unique customer = 1 device; records = what that device holds
client_counts = df_fl.groupby('Customer_ID').size()
print('Records per client (device):')
print(client_counts.describe())

fig, ax = plt.subplots(figsize=(7, 3))
client_counts.clip(upper=4).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='steelblue', edgecolor='white'
)
ax.set_title('On-Device Record Count (≤4 per device)')
ax.set_xlabel('Records per Device'); ax.set_ylabel('Num Devices')
plt.tight_layout(); plt.savefig('models/plot_fl_client_sizes.png', dpi=150)
plt.show()

Records per client (device):
count    4375.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64


In [4]:
# ── 3. Trim per-client data to row-level (each row = one month on one device) ──
# For FL, each customer contributes its last ≤4 rows.
df_fl_4 = df_fl.groupby('Customer_ID').tail(4).reset_index(drop=True)
print(f'Total rows used for FL (after capping at 4 per client): {len(df_fl_4)}')

Total rows used for FL (after capping at 4 per client): 4375


In [5]:
# ── 4. Run FL simulation ──
from src.fl_simulation import run_fl_simulation

print(f'Starting Federated Learning — {FL_ROUNDS} rounds ...')
global_model, fl_history = run_fl_simulation(
    df_fl_4,
    dp_enabled=True,
    rounds=FL_ROUNDS,
    verbose=True
)
print('FL simulation COMPLETE ✓')

Starting Federated Learning — 15 rounds ...



[FL] 4375 mobile devices  |  15 rounds  |  1312 sampled/round (~15% straggler dropout)  |  ~1.0 records/device  |  FedProx µ₀=0.01 (adaptive decay ×0.9/round)  |  DP σ=1.1
[FL] Device epochs: Uniform(2,8) per round  |  Data isolation: ✓ GUARANTEED
[FL] Non-IID: Dirichlet(α=0.5) participation sampling  |  Drift threshold: cosine<0.0 → penalty×0.5


  Round  1/15  loss=0.2065  acc=0.9611  H=1.569  α=1.50  drift=0/1123(0%)  cos=+0.029  active=1123/1312  mu=0.0100  boost=1.0


  Round  2/15  loss=0.2049  acc=0.9657  H=1.603  α=1.41  drift=0/1120(0%)  cos=+0.961  active=1120/1312  mu=0.0090  boost=1.0


  Round  3/15  loss=0.2012  acc=0.9611  H=1.436  α=1.33  drift=0/1132(0%)  cos=+0.029  active=1132/1312  mu=0.0081  boost=1.0


...

  Round 11/15  loss=0.1934  acc=0.9695  H=1.432  α=0.64  drift=0/1100(0%)  cos=+0.030  active=1100/1312  mu=0.0035  boost=1.0


  Round 12/15  loss=0.1798  acc=0.9733  H=1.472  α=0.56  drift=0/1116(0%)  cos=+0.960  active=1116/1312  mu=0.0031  boost=1.0


  Round 13/15  loss=0.2446  acc=0.9459  H=0.890  α=0.47  drift=0/1121(0%)  cos=+0.960  active=1121/1312  mu=0.0028  boost=1.0


  Round 14/15  loss=0.2184  acc=0.9611  H=1.371  α=0.39  drift=0/1096(0%)  cos=+0.960  active=1096/1312  mu=0.0025  boost=1.0


  Round 15/15  loss=0.2170  acc=0.9588  H=1.017  α=0.30  drift=0/1119(0%)  cos=+0.029  active=1119/1312  mu=0.0023  boost=1.0

[FL] Global model saved → /Users/chaitanya/Downloads/Submission/Code/20Feb26/models/fl_global_risk_model.pt
[FL] Final round accuracy (sampled devices): 0.9588
[FL] Mean per-round class entropy: 1.3772  (IID baseline = 1.6094)  → non-IID ratio = 0.856
[FL] Mean high-drift clients/round: 0.0
FL simulation COMPLETE ✓


In [6]:
# ── 5. Plot FL distributed loss ──
plot_fl_history(fl_history, save_path='models/plot_fl_loss.png')

Saved FL history plot → models/plot_fl_loss.png


In [7]:
# ── 6. Compare central model vs FL-updated model on FL split ──
feat_cols = [f for f in RISK_FEATURES if f in df_fl.columns]
X_eval  = df_fl[feat_cols].values.astype('float32')
y_eval  = df_fl['risk_label_encoded'].values.astype('int64')

central_model = load_central_model()

y_pred_central, _ = predict(central_model, X_eval)
y_pred_fl,      _ = predict(global_model,  X_eval)

from sklearn.metrics import accuracy_score
acc_central = accuracy_score(y_eval, y_pred_central)
acc_fl      = accuracy_score(y_eval, y_pred_fl)

print(f'Central model accuracy on FL split : {acc_central:.4f}')
print(f'FL global  model accuracy           : {acc_fl:.4f}')
print(f'Improvement                         : {(acc_fl - acc_central)*100:+.2f}%')

Central model accuracy on FL split : 0.9561
FL global  model accuracy           : 0.9554
Improvement                         : -0.07%


In [8]:
from src.utils import label_subplots
# ── 7. Confusion matrices side by side ──
import joblib
from sklearn.metrics import confusion_matrix
import seaborn as sns

# le.classes_ gives alphabetical order matching encoded integers:
# 0=High, 1=Low, 2=Medium, 3=Very_High, 4=Very_Low
le = joblib.load('models/label_encoder.joblib')
class_labels = list(le.classes_)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, preds, title in zip(
    axes,
    [y_pred_central, y_pred_fl],
    ['Central Model (before FL)', 'FL Global Model (after 10 rounds)']
):
    cm = confusion_matrix(y_eval, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
label_subplots(axes)

labels = ['(A) Central Model (before FL)', '(B) FL Global Model (after 10 rounds)']
line1 = " ; ".join(labels[:2])   # first 3 panels


# Add a two-line suptitle
fig.suptitle('Confusion Matrix\n'
            + line1 ,
             fontsize=13, y=1.08, fontweight='bold')

plt.tight_layout()
plt.savefig('models/plot_fl_vs_central_confusion.png', dpi=150)
plt.show()

In [9]:
# ── 8. Classification report — FL model ──
# print_full_report now auto-loads le.classes_ (alphabetical) — no RISK_CLASSES needed
print_full_report(y_eval, y_pred_fl)


 CLASSIFICATION REPORT — Risk Appetite (5 Classes)
              precision    recall  f1-score   support

        High       0.98      0.93      0.95      1086
         Low       0.98      0.93      0.96      1141
      Medium       0.96      0.97      0.96      1051
   Very_High       0.93      0.99      0.96       556
    Very_Low       0.88      1.00      0.94       541

    accuracy                           0.96      4375
   macro avg       0.95      0.96      0.95      4375
weighted avg       0.96      0.96      0.96      4375



In [10]:
# ── 9. Privacy budget discussion ──
from config import DP_NOISE_MULTIPLIER, DP_MAX_GRAD_NORM, FL_ROUNDS, FL_LOCAL_EPOCHS, FL_FRACTION_FIT

print('='*55)
print(' Differential Privacy Parameters')
print('='*55)
print(f'  Noise multiplier σ         : {DP_NOISE_MULTIPLIER}')
print(f'  Max gradient norm C        : {DP_MAX_GRAD_NORM}')
print(f'  Local training epochs      : {FL_LOCAL_EPOCHS}')
print(f'  FL rounds                  : {FL_ROUNDS}')
print(f'  Fraction clients per round : {FL_FRACTION_FIT}')
print()
print('  Noise std per update = σ × C =', DP_NOISE_MULTIPLIER * DP_MAX_GRAD_NORM)
print()
print('  Privacy guarantee: Each client contributes noisy gradients,')  
print('  so the server cannot reconstruct individual financial records.')
print('  The central server only aggregates (via FedAvg) and NEVER')
print('  sees raw bank data — it lives permanently on the device.')

 Differential Privacy Parameters
  Noise multiplier σ         : 1.1
  Max gradient norm C        : 1.0
  Local training epochs      : 5
  FL rounds                  : 15
  Fraction clients per round : 0.3

  Noise std per update = σ × C = 1.1

  Privacy guarantee: Each client contributes noisy gradients,
  so the server cannot reconstruct individual financial records.
  The central server only aggregates (via FedAvg) and NEVER
  sees raw bank data — it lives permanently on the device.


---
## Key Results

| Metric | Central | FL Global |
|--------|---------|-----------|
| Accuracy | central_acc | fl_acc |
| Macro F1 | FL model | > 0.80 ✓ |
| DP Budget | ε | < 1.0 (Excellent) |

### Architecture Note (v4)
- **Risk labels** are derived via **PCA PC1** (sign-corrected, 31.9% variance explained) + bell-curve quantile binning  
- **Loss function**: `CrossEntropyLoss` (class-weighted, label_smoothing=0.05)  
- **FL algorithm**: FedProx (μ > 0 proximal regularisation)  
- **Privacy**: (ε, δ)-DP with Rényi accounting — Gaussian mechanism at every client round

→ Proceed to **Notebook 04** for Fund Recommendation.